# Week 5 — Connect-4, Minimax, and α-β Pruning

**Lesson plan:** [`../weeks/week-05.md`](../weeks/week-05.md) · **Slides:** [`../slides/week-05/deck.md`](../slides/week-05/deck.md)

Live coding, 25 minutes. Build the game, then minimax, then α-β — instrumenting
throughout.

**Three demonstrations to reach:** same answer with fewer nodes · move ordering
compounding the win · the horizon effect losing a game in front of the class.

The agent you build here plays in the **week 5B class tournament**.

## 1 · The board

7×6 Connect-4. Bitboard-free, deliberately readable — students need to modify this
in the studio, and clarity beats cleverness at this scale.

`push`/`pop` mutate in place rather than copying. That matters: at depth 7 a copying
implementation allocates hundreds of thousands of boards and dominates the runtime.

In [ ]:
import time
from math import inf

ROWS, COLS, WIN = 6, 7, 4
MAX, MIN = 1, -1        # MAX is the search's own side


class Connect4:
    def __init__(self):
        self.grid = [[0] * COLS for _ in range(ROWS)]
        self.heights = [0] * COLS      # how many discs in each column
        self.last = None

    # ---- mechanics ---------------------------------------------------
    def legal_moves(self):
        return [c for c in range(COLS) if self.heights[c] < ROWS]

    def push(self, col, player):
        r = self.heights[col]
        self.grid[r][col] = player
        self.heights[col] += 1
        self.last = (r, col, player)

    def pop(self, col):
        self.heights[col] -= 1
        r = self.heights[col]
        self.grid[r][col] = 0

    def full(self):
        return all(h == ROWS for h in self.heights)

    # ---- terminal test ----------------------------------------------
    def _line(self, r, c, dr, dc):
        p = self.grid[r][c]
        if p == 0:
            return None
        for k in range(1, WIN):
            rr, cc = r + dr * k, c + dc * k
            if not (0 <= rr < ROWS and 0 <= cc < COLS) or self.grid[rr][cc] != p:
                return None
        return p

    def winner(self):
        for r in range(ROWS):
            for c in range(COLS):
                if self.grid[r][c] == 0:
                    continue
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    w = self._line(r, c, dr, dc)
                    if w:
                        return w
        return None

    # ---- evaluation --------------------------------------------------
    def _windows(self):
        for r in range(ROWS):
            for c in range(COLS):
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    rr, cc = r + dr * (WIN - 1), c + dc * (WIN - 1)
                    if 0 <= rr < ROWS and 0 <= cc < COLS:
                        yield [self.grid[r + dr * k][c + dc * k] for k in range(WIN)]

    def evaluate(self):
        """Heuristic value from MAX's point of view."""
        w = self.winner()
        if w == MAX:
            return 10_000
        if w == MIN:
            return -10_000
        score = 0
        for win in self._windows():
            mx, mn = win.count(MAX), win.count(MIN)
            if mx and mn:
                continue                       # blocked, worthless to both
            if mx:
                score += {1: 1, 2: 10, 3: 50}[mx]
            elif mn:
                score -= {1: 1, 2: 10, 3: 50}[mn]
        # central columns are worth more -- classic Connect-4 domain knowledge
        for r in range(ROWS):
            if self.grid[r][COLS // 2] == MAX:
                score += 3
            elif self.grid[r][COLS // 2] == MIN:
                score -= 3
        return score

    def __str__(self):
        sym = {0: ".", MAX: "X", MIN: "O"}
        rows = ["|" + " ".join(sym[self.grid[r][c]] for c in range(COLS)) + "|"
                for r in reversed(range(ROWS))]
        return "\n".join(rows) + "\n " + " ".join(str(c) for c in range(COLS))


b = Connect4()
for col, p in [(3, MAX), (3, MIN), (4, MAX), (2, MIN)]:
    b.push(col, p)
print(b)
print("\nevaluate() =", b.evaluate(), " legal:", b.legal_moves())

## 2 · Minimax

The recursion is the definition of the game value: MAX maximizes, MIN minimizes,
and there is no assumption of opponent error anywhere in it.

In [ ]:
def minimax(board, depth, player, counter):
    counter[0] += 1
    winner = board.winner()
    if winner is not None or depth == 0 or board.full():
        return board.evaluate(), None

    best_move = None
    if player == MAX:
        best = -inf
        for move in board.legal_moves():
            board.push(move, MAX)
            value, _ = minimax(board, depth - 1, MIN, counter)
            board.pop(move)
            if value > best:
                best, best_move = value, move
        return best, best_move
    else:
        best = inf
        for move in board.legal_moves():
            board.push(move, MIN)
            value, _ = minimax(board, depth - 1, MAX, counter)
            board.pop(move)
            if value < best:
                best, best_move = value, move
        return best, best_move


board = Connect4()
board.push(3, MAX); board.push(3, MIN)
c = [0]
t0 = time.perf_counter()
v, m = minimax(board, 4, MAX, c)
print(f"depth 4: value={v} move={m} nodes={c[0]:,} time={time.perf_counter()-t0:.3f}s")

## 3 · α-β pruning

One `break`. That is the entire optimization.

`alpha` is the best value MAX can already guarantee; `beta` is the best MIN can
already guarantee. When they cross, the rest of this branch cannot affect the
result — so we stop looking.

In [ ]:
def alphabeta(board, depth, player, alpha, beta, counter, order=None):
    counter[0] += 1
    winner = board.winner()
    if winner is not None or depth == 0 or board.full():
        return board.evaluate(), None

    moves = board.legal_moves()
    if order is not None:
        moves = [c for c in order if c in moves]

    best_move = None
    if player == MAX:
        best = -inf
        for move in moves:
            board.push(move, MAX)
            value, _ = alphabeta(board, depth - 1, MIN, alpha, beta, counter, order)
            board.pop(move)
            if value > best:
                best, best_move = value, move
            alpha = max(alpha, best)
            if alpha >= beta:
                break                      # <-- the entire optimization
        return best, best_move
    else:
        best = inf
        for move in moves:
            board.push(move, MIN)
            value, _ = alphabeta(board, depth - 1, MAX, alpha, beta, counter, order)
            board.pop(move)
            if value < best:
                best, best_move = value, move
            beta = min(beta, best)
            if alpha >= beta:
                break
        return best, best_move

## 4 · Demonstration 1 — same answer, far fewer nodes

⚠️ **Pruning is not approximation.** α-β returns the *identical* value minimax
returns. It just refuses to examine branches that provably cannot matter.

The assertion below is the important line in this notebook.

In [ ]:
def fresh():
    b = Connect4()
    b.push(3, MAX); b.push(3, MIN)
    return b


print(f"  {'depth':>6}{'minimax':>12}{'alpha-beta':>13}{'speedup':>10}{'same value?':>13}")
print("  " + "-" * 55)
for depth in (2, 4, 5, 6):
    b1, c1 = fresh(), [0]
    t0 = time.perf_counter()
    v1, _ = minimax(b1, depth, MAX, c1)
    t1 = time.perf_counter() - t0

    b2, c2 = fresh(), [0]
    t0 = time.perf_counter()
    v2, _ = alphabeta(b2, depth, MAX, -inf, inf, c2)
    t2 = time.perf_counter() - t0

    assert v1 == v2, f"VALUES DIFFER at depth {depth}: {v1} vs {v2}"
    print(f"  {depth:>6}{c1[0]:>12,}{c2[0]:>13,}{c1[0]/c2[0]:>9.1f}x{'yes':>13}")

print("\nAll values identical. Pruning changed the COST, not the ANSWER.")

## 5 · Demonstration 2 — move ordering compounds the win

α-β's power depends entirely on examining good moves first. In Connect-4, central
columns are strong, so `3, 2, 4, 1, 5, 0, 6` is a decent static ordering.

Theory: perfect ordering gives `b^(m/2)` — **double the searchable depth**. Random
ordering gives roughly `b^(3m/4)`. Watch the middle column of the table.

In [ ]:
CENTER_FIRST = [3, 2, 4, 1, 5, 0, 6]
LEFT_TO_RIGHT = [0, 1, 2, 3, 4, 5, 6]
WORST = [0, 6, 1, 5, 2, 4, 3]

print(f"  {'depth':>6}{'left-to-right':>15}{'center-first':>14}{'gain':>8}")
print("  " + "-" * 44)
for depth in (4, 5, 6, 7):
    _, cl = None, [0]
    alphabeta(fresh(), depth, MAX, -inf, inf, cl, LEFT_TO_RIGHT)
    _, cc = None, [0]
    alphabeta(fresh(), depth, MAX, -inf, inf, cc, CENTER_FIRST)
    print(f"  {depth:>6}{cl[0]:>15,}{cc[0]:>14,}{cl[0]/cc[0]:>7.2f}x")

print("""
One line -- a sorted move list -- and the node count drops again. Ordering is not
a micro-optimization; it decides how deep you can afford to look, and depth is
strength. This is why real engines spend effort on move ordering before anything
else.""")

## 6 · Demonstration 3 — the horizon effect

The one students remember.

A **depth-4** agent cannot see a threat that resolves at depth 5. It plays a move
that looks fine by its own evaluation and loses. The evaluation function is not
wrong; the *horizon* is.

In [ ]:
def play(depth_max, depth_min, order=CENTER_FIRST, max_plies=42, verbose=False):
    """MAX vs MIN, both alpha-beta, different depths. Returns winner."""
    board = Connect4()
    player = MAX
    for ply in range(max_plies):
        if board.winner() is not None or board.full():
            break
        d = depth_max if player == MAX else depth_min
        _, move = alphabeta(board, d, player, -inf, inf, [0], order)
        if move is None:
            break
        board.push(move, player)
        if verbose:
            print(f"ply {ply}: {'MAX' if player==MAX else 'MIN'} -> col {move}")
        player = -player
    return board.winner(), board


for dmax, dmin in [(2, 5), (4, 5), (5, 5), (6, 4)]:
    w, board = play(dmax, dmin)
    name = {MAX: "MAX wins", MIN: "MIN wins", None: "draw"}[w]
    print(f"MAX depth {dmax} vs MIN depth {dmin}:  {name}")

print("""
Depth is strength -- MOSTLY. See the note below on the 5-vs-5 row.
The shallow agent is not making
"mistakes" it could detect -- every move it plays is optimal WITH RESPECT TO ITS
HORIZON. It simply cannot see far enough.

That is the horizon effect, and no evaluation function fixes it. Only more depth
does -- which is exactly what alpha-beta buys you.""")

### ⚠️ One result in that table is an artifact — say so

`MAX depth 5 vs MIN depth 5` → **MIN wins.** At *equal* depth the second player
won. That is not a fact about Connect-4: with perfect play the **first** player
wins. It is an artifact of our evaluation function and of MAX having to commit
first with imperfect information.

Two things follow, and both are course-level lessons:

1. **Depth is strength only relative to a fixed evaluation function.** The 6-vs-4
   result is the real signal; the 5-vs-5 result tells you about `evaluate()`, not
   about search.
2. **Do not let a tidy story survive contact with your own output.** The
   monotonic-depth claim is *mostly* true here and I have flagged where it is not.

Ask the class to explain the 5-vs-5 row before you explain it. It is a better
diagnostic of who understands the evaluation/horizon distinction than any exam
question.

## 7 · Watch a shallow agent lose a specific game

Print the final board of the depth-2 vs depth-5 match. This is the artifact to
show the class.

In [ ]:
w, board = play(2, 5)
print(board)
print(f"\nwinner: {'MAX (X)' if w == MAX else 'MIN (O)' if w == MIN else 'draw'}")
print("""
X is the depth-2 agent. Trace back and find the ply where it allowed a threat it
could not see. Every one of its moves was locally reasonable.""")

## 8 · Minimax assumes a perfect opponent

Worth stating explicitly, because it is the assumption that breaks in the
tournament: minimax is **pessimistic**. It assumes MIN always plays the best reply.

Against a *weak* opponent this is suboptimal — it will decline a risky move with a
huge payoff that a fallible opponent would very likely fail to punish.

> **Modeling your opponent as perfect is safe, not optimal.** Ask which one you
> actually want in the tournament.

## 9 · Studio preview

- Add **transposition tables** — the same position reached by different move
  orders is currently searched twice.
- Add **iterative deepening** so the agent respects a *time* budget rather than a
  depth budget. This is what the tournament enforces.
- Then the week's comparison: an **LLM playing Connect-4** from a board
  description. Score legality first, strength second. Expect illegal moves — those
  go in the Failure Atlas.

Shannon's 1950 type A (brute force) vs type B (selective) framing is the right
lens, and the LLM is neither: it is a **learned move prior with no search at
all**. Ruoss et al. (2024) show amortized planning can reach grandmaster chess
without search — so the interesting question is not "can it play" but **"where did
the search go?"**